In [1]:
# Load and parse pits, then create the manuscript ECTP slabs.
from pathlib import Path
import sys
import warnings

# Works from paper_analysis/, examples/, the repository, or the Snow workspace.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / "SnowPyt-MechParams"]
repo_root = next(p for p in candidates if (p / "src/snowpyt_mechparams").is_dir())
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "examples"))

from notebook_utils import load_pits, create_ectp_slabs

# Each slab contains the layers above the ECTP failure layer. Multiple valid
# ECTP results in the same pit remain separate slabs, even at the same depth.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Using UFloat objects with std_dev==0.*", category=UserWarning)
    pits = load_pits(str(repo_root / "examples/data"))
    ectp_slabs = create_ectp_slabs(pits)

assert len(pits) == 50_278, "The input dataset differs from the manuscript."
assert len(ectp_slabs) == 14_776, "The ECTP cohort differs from the manuscript."
assert len({slab.slab_id for slab in ectp_slabs}) == len(ectp_slabs)
print(f"Loaded and parsed {len(pits):,} snow pits.")
print(f"Created {len(ectp_slabs):,} ECTP slabs.")


Loaded and parsed 50,278 snow pits.
Created 14,776 ECTP slabs.


In [2]:
# How many pits account for the 14,776 slabs?
from collections import Counter

# Count only pits that actually yielded a slab in the analysis cohort.
# A pit with an ECT result that could not form a valid slab is not included.
assert all(slab.pit_id is not None and slab.pit_id != "" for slab in ectp_slabs)
slabs_per_pit = Counter(slab.pit_id for slab in ectp_slabs)
n_pits_with_ectp_slabs = len(slabs_per_pit)
n_multi_slab_pits = sum(count > 1 for count in slabs_per_pit.values())
pit_count_by_slab_count = Counter(slabs_per_pit.values())

assert sum(slabs_per_pit.values()) == len(ectp_slabs)
assert sum(n_slabs * n_pits for n_slabs, n_pits in pit_count_by_slab_count.items()) == len(ectp_slabs)
print(f"{n_pits_with_ectp_slabs:,} distinct pits yielded the {len(ectp_slabs):,} ECTP slabs.")
print(f"Pits yielding multiple ECTP slabs: {n_multi_slab_pits:,}")
print(f"Mean ECTP slabs per contributing pit: {len(ectp_slabs) / n_pits_with_ectp_slabs:.2f}")
print("\nECTP slabs per pit | Number of pits | Total slabs")
for n_slabs, n_pits in sorted(pit_count_by_slab_count.items()):
    print(f"{n_slabs:>18,} | {n_pits:>14,} | {n_slabs * n_pits:>11,}")


12,347 distinct pits yielded the 14,776 ECTP slabs.
Pits yielding multiple ECTP slabs: 2,113
Mean ECTP slabs per contributing pit: 1.20

ECTP slabs per pit | Number of pits | Total slabs
                 1 |         10,234 |      10,234
                 2 |          1,858 |       3,716
                 3 |            205 |         615
                 4 |             40 |         160
                 5 |              9 |          45
                 6 |              1 |           6
